# Proyecto Final de Minería de Datos
## Notebook 03. Despliegue (Credit Card Churn)

En este notebook cargamos el Pipeline guardado en el Notebook 02, lo probamos con clientes nuevos y con datos crudos del archivo original, describimos la aplicación en Streamlit y definimos cómo se monitorearía y mantendría el modelo.


---
## 1. Verificación del Pipeline guardado

In [3]:
import joblib
import pandas as pd
import numpy as np

bundle = joblib.load("modelo_churn_pipeline.joblib")
pipeline = bundle["pipeline"]

print("=== PIPELINE CARGADO EXITOSAMENTE PARA PRODUCCIÓN ===")
print(f"Pasos del Pipeline: {[step[0] for step in pipeline.steps]}")
print(f"Método de optimización ganador: {bundle['optimization_method']}")
print(f"Hiperparámetros óptimos del clasificador XGBoost: {bundle['best_params']}")
print(f"Métricas validadas en Test (30%): {bundle['test_metrics']}")


=== PIPELINE CARGADO EXITOSAMENTE PARA PRODUCCIÓN ===
Pasos del Pipeline: ['unknown_to_nan', 'imputer', 'features', 'smote', 'preprocessor', 'classifier']
Método de optimización ganador: Optimización Bayesiana (Optuna TPE)
Hiperparámetros óptimos del clasificador XGBoost: {'n_estimators': 200, 'max_depth': 7, 'learning_rate': 0.1232, 'subsample': 0.8697, 'colsample_bytree': 0.739}
Métricas validadas en Test (30%): {'Accuracy': 0.9671, 'Precision': 0.8927, 'Recall': 0.9037, 'F1-Score': 0.8982, 'ROC-AUC': 0.9911}


---
## 2. Inferencia sobre nuevos clientes

In [4]:
nuevos_clientes = pd.DataFrame([
    {
        "Perfil": "Cliente A (Activo y Vinculado)",
        "Customer_Age": 45, "Gender": "M", "Dependent_count": 2,
        "Education_Level": "Graduate", "Marital_Status": "Married",
        "Income_Category": "$80K - $120K", "Card_Category": "Blue",
        "Months_on_book": 36, "Total_Relationship_Count": 5,
        "Months_Inactive_12_mon": 1, "Contacts_Count_12_mon": 1,
        "Credit_Limit": 12000.0, "Total_Revolving_Bal": 1450,
        "Total_Amt_Chng_Q4_Q1": 0.85, "Total_Trans_Amt": 4800,
        "Total_Trans_Ct": 78, "Total_Ct_Chng_Q4_Q1": 0.82,
        "Avg_Utilization_Ratio": 0.121
    },
    {
        "Perfil": "Cliente B (Baja Actividad y Múltiples Reclamos)",
        "Customer_Age": 48, "Gender": "F", "Dependent_count": 3,
        "Education_Level": "High School", "Marital_Status": "Single",
        "Income_Category": "Less than $40K", "Card_Category": "Blue",
        "Months_on_book": 39, "Total_Relationship_Count": 2,
        "Months_Inactive_12_mon": 4, "Contacts_Count_12_mon": 4,
        "Credit_Limit": 3200.0, "Total_Revolving_Bal": 0,
        "Total_Amt_Chng_Q4_Q1": 0.42, "Total_Trans_Amt": 1650,
        "Total_Trans_Ct": 32, "Total_Ct_Chng_Q4_Q1": 0.38,
        "Avg_Utilization_Ratio": 0.000
    }
])

X_nuevos = nuevos_clientes.drop(columns=["Perfil"])
preds = pipeline.predict(X_nuevos)
probs = pipeline.predict_proba(X_nuevos)[:, 1]

res_inferencia = nuevos_clientes[["Perfil", "Total_Trans_Ct", "Total_Revolving_Bal", "Months_Inactive_12_mon", "Contacts_Count_12_mon"]].copy()
res_inferencia["Predicción"] = np.where(preds == 1, "Attrited Customer (Abandona)", "Existing Customer (Retenido)")
res_inferencia["Probabilidad_Churn (%)"] = np.round(probs * 100, 2)
print("=== PRUEBA DE INFERENCIA EN DATOS FUTUROS ===")
print(res_inferencia.to_string(index=False))


=== PRUEBA DE INFERENCIA EN DATOS FUTUROS ===
                                         Perfil  Total_Trans_Ct  Total_Revolving_Bal  Months_Inactive_12_mon  Contacts_Count_12_mon                   Predicción  Probabilidad_Churn (%)
                 Cliente A (Activo y Vinculado)              78                 1450                       1                      1 Existing Customer (Retenido)                     0.0
Cliente B (Baja Actividad y Múltiples Reclamos)              32                    0                       4                      4 Attrited Customer (Abandona)                   100.0


In [6]:
# Prueba con datos crudos del CSV original, que traen Unknown en tres columnas
df_original = pd.read_csv("credit_card_churn.csv")
columnas_unknown = ['Education_Level', 'Marital_Status', 'Income_Category']
filas_unknown = pd.concat([df_original[df_original[c] == 'Unknown'].head(2) for c in columnas_unknown]).drop_duplicates().head(5)
X_unknown = filas_unknown[bundle["raw_columns"]]
assert all((X_unknown[c] == 'Unknown').any() for c in columnas_unknown)

res_unknown = filas_unknown[columnas_unknown + ['Attrition_Flag']].copy()
res_unknown["Predicción"] = np.where(pipeline.predict(X_unknown) == 1, "Attrited Customer", "Existing Customer")
res_unknown["Probabilidad_Churn (%)"] = np.round(pipeline.predict_proba(X_unknown)[:, 1] * 100, 2)
print("=== PRUEBA CON FILAS CRUDAS DEL CSV ORIGINAL CON Unknown ===")
print(res_unknown.to_string())

# El archivo completo tampoco debe fallar ni producir probabilidades inválidas
X_completo = df_original[bundle["raw_columns"]]
probs_completo = pipeline.predict_proba(X_completo)[:, 1]
assert np.isfinite(probs_completo).all() and ((probs_completo >= 0) & (probs_completo <= 1)).all()
print(f"\nArchivo completo: {len(X_completo)} filas, {int((X_completo == 'Unknown').any(axis=1).sum())} con Unknown, todas calificadas")

=== PRUEBA CON FILAS CRUDAS DEL CSV ORIGINAL CON Unknown ===
   Education_Level Marital_Status Income_Category     Attrition_Flag         Predicción  Probabilidad_Churn (%)
6          Unknown        Married         $120K +  Existing Customer  Existing Customer                    0.02
11         Unknown        Married     $40K - $60K  Existing Customer  Existing Customer                    0.00
3      High School        Unknown  Less than $40K  Existing Customer  Existing Customer                    0.05
7      High School        Unknown     $60K - $80K  Existing Customer  Existing Customer                    0.03
19        Graduate        Married         Unknown  Existing Customer  Existing Customer                    0.00

Archivo completo: 10127 filas, 3046 con Unknown, todas calificadas


---
## 3. Aplicación en Streamlit
La interfaz está en `app_streamlit.py` y tiene dos pestañas. En la primera se ingresan los datos de un cliente y la app muestra la probabilidad de que abandone, junto con una recomendación comercial. En la segunda se sube un archivo CSV con varios clientes y la app devuelve la predicción de cada uno, con la opción de descargar los resultados.

Para ejecutarla localmente se usa el comando `streamlit run app_streamlit.py` desde la carpeta del proyecto.


---
## 4. Monitoreo y mantenimiento

### Valores de referencia
Tomamos como referencia el desempeño del modelo en el 30% de prueba, con F1 de 0.8841, Recall de 0.8914, Precision de 0.8770 y ROC-AUC de 0.9905. En ese conjunto el modelo alerta sobre el 16.34% de los clientes, muy cerca de la tasa real de abandono (16.07%). Estos umbrales son una propuesta inicial y deberían ajustarse cuando haya datos reales de uso.

### Cambios en los datos
Cada mes se compara la distribución de las variables nuevas con la del entrenamiento, empezando por Total_Trans_Ct y Total_Trans_Amt, que son las más importantes del modelo. Para eso se puede usar el PSI, un índice que mide cuánto cambió una variable (menos de 0.10 es estable, entre 0.10 y 0.25 hay que revisar y más de 0.25 es un cambio fuerte). También se vigila si aumenta el porcentaje de Unknown o si aparecen categorías que el modelo no conocía.

### Cambios en el desempeño
Como el abandono solo se confirma tiempo después, las métricas reales se calculan cuando ya se sabe qué pasó con cada cliente. Se genera una alerta si el Recall baja de 0.85 (el mínimo de la línea base del informe) o si el F1 baja de 0.86. Mientras tanto, un cambio fuerte en la tasa de alertas frente al 16.34% puede indicar que algo cambió.

### Cronograma de mantenimiento

| Frecuencia | Actividad |
| :--- | :--- |
| Mensual | Revisar el PSI, el porcentaje de Unknown, las categorías nuevas y la tasa de alertas |
| Trimestral | Calcular Recall, Precision, F1 y ROC-AUC con los clientes cuyo desenlace ya se conoce |
| Anual | Reentrenar el modelo con datos nuevos y compararlo con el actual |
| Por evento | Reentrenar antes si se supera algún umbral o si cambian los productos o las políticas del banco |

Para reentrenar se repiten los notebooks 01 y 02 con los datos actualizados, y el modelo nuevo solo reemplaza al actual si es mejor en un conjunto de prueba reciente. El archivo `churn_features.py` debe ir siempre junto al modelo, porque el Pipeline depende de él.